**Credit Card Fraud Detection: Precision vs. Recall at Different Alert Thresholds**

_How many false alarms can a fraud model avoid by raising its alert threshold, and how much fraud does it miss in exchange?_

Facundo Urquizo - Statistics and Quantitative Modeling - Data Science Track (Baruch College - Zicklin School of Business)

In [ ]:
#Importing methods and libraries
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, average_precision_score, precision_recall_curve
import matplotlib.pyplot as plt
import pandas as pd
import zipfile
#Unziping file to open CSV
with zipfile.ZipFile('creditcard.zip', 'r') as z:
    creditcard_file = pd.read_csv(z.open('creditcard.csv'))
creditcard_file.head()

Splitting the data: 60% training, 20% validation, 20% test

In [ ]:
X = creditcard_file.drop(columns=['Class'])
y = creditcard_file['Class']
#Stratify keeps the same share of fraud (about 0.17%) in every split. Without it the number of frauds in the test set depends on luck
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=2)
#The validation set is for choosing the threshold. The test set is only used once at the end
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=2)
print(f"Frauds in training: {y_train.sum()} | validation: {y_val.sum()} | test: {y_test.sum()}")

Scaling, SMOTE and the two models

In [ ]:
#Standard Scaler: Equal mathematical weight. Fitted on the training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
#SMOTE (Synthetic Minority Over-sampling Technique): Generate artificial fraud. Only applied to the training data so validation and test keep the real fraud rate
smote = SMOTE(random_state=4)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
#Logistic Regression: a simple baseline to check whether the random forest is actually better
baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_smote, y_train_smote)
#Random Forest Classifier: Evaluate Transactions
classifier = RandomForestClassifier(random_state=42, n_jobs=-1)
classifier.fit(X_train_smote, y_train_smote)

Choosing the threshold on the validation set

In [ ]:
#Fraud score for every validation transaction
val_probabilities = classifier.predict_proba(X_val_scaled)[:, 1]
#Precision and recall at each candidate threshold
rows = []
for t in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    guesses = (val_probabilities >= t).astype(int)
    rows.append({'threshold': t,
                 'alerts': guesses.sum(),
                 'precision': precision_score(y_val, guesses, zero_division=0),
                 'recall': recall_score(y_val, guesses)})
threshold_table = pd.DataFrame(rows)
print(threshold_table.round(3).to_string(index=False))
#Rule: take the most precise threshold that still catches at least 80% of the validation fraud
#If none of them reach 80% recall, stay at the default 0.50
min_recall = 0.80
allowed = threshold_table[threshold_table['recall'] >= min_recall]
if len(allowed) > 0:
    chosen_threshold = allowed.sort_values(['precision', 'threshold']).iloc[-1]['threshold']
else:
    chosen_threshold = 0.50
print(f"Chosen threshold: {chosen_threshold:.2f}")

Results on the test set (not used for any decision above)

In [ ]:
test_probabilities = classifier.predict_proba(X_test_scaled)[:, 1]
default_guesses = (test_probabilities >= 0.50).astype(int)
strict_guesses = (test_probabilities >= chosen_threshold).astype(int)
print("Default Threshold (50%) Scorecard:")
print(classification_report(y_test, default_guesses))
print(f"Strict Threshold ({chosen_threshold:.0%}) Scorecard:")
print(classification_report(y_test, strict_guesses))
#Counts instead of percentages, since the test set only has about 100 frauds and one transaction moves the percentages by a point
for name, guesses in [("50%", default_guesses), (f"{chosen_threshold:.0%}", strict_guesses)]:
    tn, fp, fn, tp = confusion_matrix(y_test, guesses).ravel()
    print(f"Threshold {name}: caught {tp} frauds, missed {fn}, false alarms {fp}")

In [ ]:
#PR-AUC (average precision) summarizes every threshold at once. Accuracy and ROC-AUC look great on this data no matter what because 99.8% of transactions are normal
baseline_probabilities = baseline.predict_proba(X_test_scaled)[:, 1]
print(f"PR-AUC Random Forest: {average_precision_score(y_test, test_probabilities):.3f}")
print(f"PR-AUC Logistic Regression: {average_precision_score(y_test, baseline_probabilities):.3f}")
print(f"PR-AUC of random guessing: {y_test.mean():.4f}")
#Precision-recall curves for both models
fig, ax = plt.subplots(figsize=(10, 6))
for name, scores in [("Random Forest", test_probabilities), ("Logistic Regression", baseline_probabilities)]:
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ax.plot(recall, precision, label=name)
ax.set_xlabel('Recall (share of fraud caught)')
ax.set_ylabel('Precision (share of alerts that are fraud)')
ax.set_title('Precision-Recall Curve on the Test Set')
ax.grid(True)
ax.legend()
plt.show()